# Distributed Workflow Guide

This notebook is designed to work across two machines:
1. **High-RAM Machine**: For data preprocessing and feature engineering
2. **Low-RAM Machine**: For model training and inference

## Workflow Overview:
1. Run preprocessing cells on high-RAM machine (Sections 1-3)
2. Save processed data to GitHub
3. Run model training and inference on low-RAM machine (Sections 4-10)

See specific instructions below for each machine.

# Network Attack Detection using Graph Neural Networks

This notebook implements a Graph Neural Network (GNN) based approach for network attack detection. We'll compare the performance between using all features vs. the top 20 most important features.

# 1. Import Dependencies and Utilities

# High-RAM Machine Instructions

The following sections (1-3) should be run on the machine with more RAM. These sections handle:
- Data loading from multiple large CSV files
- Feature engineering and selection
- Data preprocessing and scaling

Output files will be saved to these locations:
- `playground/models/selected_features.json`
- `playground/models/label_mapping.json`
- `playground/scalers/scaler_20.pkl`
- `playground/scalers/scaler_full.pkl`
- `playground/processed_data/train_20.npz`
- `playground/processed_data/val_20.npz`
- `playground/processed_data/test_20.npz`
- `playground/processed_data/train_full.npz`
- `playground/processed_data/val_full.npz`
- `playground/processed_data/test_full.npz`

After running these sections, commit and push these files to GitHub.

In [2]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
from datetime import datetime
import json

# ML libraries
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif

# PyTorch and PyTorch Geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, global_mean_pool

# Custom utilities
import sys
sys.path.append('..')
from playground.playground_utils import load_and_concatenate_datasets, inspect_missing_and_constant

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

c:\Users\yuheng\Documents\fyp\backend\fypenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


# 2. Data Loading and Preprocessing

First, we'll load and combine the datasets from testDataSet directory. We'll handle missing values and encode categorical features.

In [2]:
# Define dataset paths
DATASET_DIR = '../testDataSet'
training_files = [
    '03-01-2018.csv',
    'combined_cicids.csv',
    'DrDoS_DNS_data_1_per.csv',
    'Friday-16-02-2018_TrafficForML_CICFlowMeter.csv'
]

# Function to load and preprocess dataset
def load_and_preprocess_data(file_paths, dataset_dir):
    dfs = []
    for file in file_paths:
        path = os.path.join(dataset_dir, file)
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Loaded {file}: {df.shape}")
            dfs.append(df)
        else:
            print(f"File not found: {file}")
    
    # Combine all datasets
    combined_df = pd.concat(dfs, ignore_index=True)
    print(f"\nCombined shape: {combined_df.shape}")
    
    return combined_df

# Load the datasets
df = load_and_preprocess_data(training_files, DATASET_DIR)

# Display dataset statistics
print("\nClass distribution:")
print(df['Label'].value_counts())

C:\Users\yuheng\AppData\Local\Temp\ipykernel_2916\951250571.py:16: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Loaded 03-01-2018.csv: (331125, 80)


C:\Users\yuheng\AppData\Local\Temp\ipykernel_2916\951250571.py:16: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Loaded combined_cicids.csv: (1379700, 80)


C:\Users\yuheng\AppData\Local\Temp\ipykernel_2916\951250571.py:16: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Loaded DrDoS_DNS_data_1_per.csv: (5074413, 88)


C:\Users\yuheng\AppData\Local\Temp\ipykernel_2916\951250571.py:16: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Loaded Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: (1048575, 80)

Combined shape: (7833813, 165)

Combined shape: (7833813, 165)


KeyboardInterrupt: 

In [ ]:
# Data preprocessing
def preprocess_data(df):
    # Handle missing values
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(df.mean())
    
    # Convert categorical variables
    label_encoder = LabelEncoder()
    df['Label'] = label_encoder.fit_transform(df['Label'])
    
    # Save label mapping
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
    with open('../playground/models/label_mapping.json', 'w') as f:
        json.dump(label_mapping, f)
    
    print("Label mapping:", label_mapping)
    return df, label_mapping

# Preprocess the data
df_processed, label_mapping = preprocess_data(df)

# Split features and labels
X = df_processed.drop(['Label'], axis=1)
y = df_processed['Label']

# Split into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"\nData split sizes:")
print(f"Training: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")

# 3. Feature Selection and Scaling

We'll implement feature selection for the top 20 features and create scalers for both feature sets.

In [ ]:
# Feature selection for top 20 features
def select_top_features(X_train, y_train, X_val, X_test, k=20):
    selector = SelectKBest(score_func=f_classif, k=k)
    X_train_selected = selector.fit_transform(X_train, y_train)
    
    # Get selected feature names
    selected_features = X_train.columns[selector.get_support()].tolist()
    print(f"\nTop {k} selected features:")
    for i, feature in enumerate(selected_features, 1):
        print(f"{i}. {feature}")
    
    # Transform validation and test sets
    X_val_selected = selector.transform(X_val)
    X_test_selected = selector.transform(X_test)
    
    # Save selected features
    with open('../playground/models/selected_features.json', 'w') as f:
        json.dump(selected_features, f)
    
    return X_train_selected, X_val_selected, X_test_selected, selected_features

# Scale the features
def scale_features(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

# Get both feature sets
X_train_20, X_val_20, X_test_20, selected_features = select_top_features(X_train, y_train, X_val, X_test)

# Scale both feature sets
X_train_20_scaled, X_val_20_scaled, X_test_20_scaled, scaler_20 = scale_features(X_train_20, X_val_20, X_test_20)
X_train_full_scaled, X_val_full_scaled, X_test_full_scaled, scaler_full = scale_features(X_train, X_val, X_test)

# Save scalers
import joblib
joblib.dump(scaler_20, '../playground/scalers/scaler_20.pkl')
joblib.dump(scaler_full, '../playground/scalers/scaler_full.pkl')

In [ ]:
# Create directory for processed data
os.makedirs('../playground/processed_data', exist_ok=True)

# Save processed datasets
def save_processed_data(X, y, name):
    """Save processed features and labels"""
    np.savez_compressed(
        f'../playground/processed_data/{name}.npz',
        features=X,
        labels=y
    )
    print(f"Saved {name} data with shape: {X.shape}")

# Save training data
save_processed_data(X_train_20_scaled, y_train.values, 'train_20')
save_processed_data(X_val_20_scaled, y_val.values, 'val_20')
save_processed_data(X_test_20_scaled, y_test.values, 'test_20')

save_processed_data(X_train_full_scaled, y_train.values, 'train_full')
save_processed_data(X_val_full_scaled, y_val.values, 'val_full')
save_processed_data(X_test_full_scaled, y_test.values, 'test_full')

print("\nAll processed data has been saved. You can now commit and push these files to GitHub.")

# 4. Graph Construction

Define functions to construct graph representations from network flow data.

# Low-RAM Machine Instructions

The following sections (4-10) should be run on the machine with less RAM. These sections handle:
- Graph construction from preprocessed data
- Model training and evaluation
- Performance comparison
- Model inference and retraining

Before running these sections:
1. Pull the latest changes from GitHub
2. Ensure all files from the high-RAM machine are present in the correct locations
3. Skip sections 1-3 as they require high RAM

First, let's load the preprocessed data:

In [ ]:
# Load preprocessed data
def load_processed_data(name):
    """Load processed features and labels"""
    data = np.load(f'../playground/processed_data/{name}.npz')
    return data['features'], data['labels']

# Load all datasets
X_train_20_scaled, y_train = load_processed_data('train_20')
X_val_20_scaled, y_val = load_processed_data('val_20')
X_test_20_scaled, y_test = load_processed_data('test_20')

X_train_full_scaled, _ = load_processed_data('train_full')
X_val_full_scaled, _ = load_processed_data('val_full')
X_test_full_scaled, _ = load_processed_data('test_full')

# Load feature information
with open('../playground/models/selected_features.json', 'r') as f:
    selected_features = json.load(f)

with open('../playground/models/label_mapping.json', 'r') as f:
    label_mapping = json.load(f)

print("Loaded preprocessed data successfully")
print(f"Training data shape (20 features): {X_train_20_scaled.shape}")
print(f"Training data shape (all features): {X_train_full_scaled.shape}")
print(f"Number of classes: {len(label_mapping)}")

In [ ]:
# Custom Dataset class for graph data
class NetworkFlowDataset(Dataset):
    def __init__(self, X, y, batch_size=32):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.batch_size = batch_size
        
    def len(self):
        return len(self.X)
    
    def get(self, idx):
        # Create a fully connected graph for each batch
        batch_x = self.X[idx:idx+self.batch_size]
        batch_y = self.y[idx:idx+self.batch_size]
        num_nodes = len(batch_x)
        
        # Create fully connected edges (all-to-all connections)
        edge_index = []
        for i in range(num_nodes):
            for j in range(num_nodes):
                if i != j:
                    edge_index.append([i, j])
        edge_index = torch.LongTensor(edge_index).t()
        
        # Create the graph data object
        data = Data(x=batch_x, 
                   edge_index=edge_index,
                   y=batch_y)
        
        return data

# Create graph datasets
train_dataset_20 = NetworkFlowDataset(X_train_20_scaled, y_train.values)
val_dataset_20 = NetworkFlowDataset(X_val_20_scaled, y_val.values)
test_dataset_20 = NetworkFlowDataset(X_test_20_scaled, y_test.values)

train_dataset_full = NetworkFlowDataset(X_train_full_scaled, y_train.values)
val_dataset_full = NetworkFlowDataset(X_val_full_scaled, y_val.values)
test_dataset_full = NetworkFlowDataset(X_test_full_scaled, y_test.values)

# Create data loaders
BATCH_SIZE = 32

train_loader_20 = DataLoader(train_dataset_20, batch_size=1, shuffle=True)
val_loader_20 = DataLoader(val_dataset_20, batch_size=1)
test_loader_20 = DataLoader(test_dataset_20, batch_size=1)

train_loader_full = DataLoader(train_dataset_full, batch_size=1, shuffle=True)
val_loader_full = DataLoader(val_dataset_full, batch_size=1)
test_loader_full = DataLoader(test_dataset_full, batch_size=1)

# 5. GNN Model Definition

Define our GNN architecture using PyTorch Geometric's GNN layers.

In [ ]:
class GNNModel(torch.nn.Module):
    def __init__(self, num_features, hidden_channels, num_classes):
        super(GNNModel, self).__init__()
        self.conv1 = GATConv(num_features, hidden_channels, heads=4)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels * 2, heads=2)
        self.conv3 = GATConv(hidden_channels * 2, hidden_channels, heads=1)
        self.lin = nn.Linear(hidden_channels, num_classes)
        
        # Batch normalization layers
        self.bn1 = nn.BatchNorm1d(hidden_channels * 4)
        self.bn2 = nn.BatchNorm1d(hidden_channels * 2)
        self.bn3 = nn.BatchNorm1d(hidden_channels)
        
        # Dropout layer
        self.dropout = nn.Dropout(0.2)
        
    def forward(self, x, edge_index, batch):
        # First Graph Attention layer
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = self.dropout(x)
        
        # Second Graph Attention layer
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        x = self.dropout(x)
        
        # Third Graph Attention layer
        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.elu(x)
        
        # Global mean pooling
        x = global_mean_pool(x, batch)
        
        # Final classification layer
        x = self.lin(x)
        
        return x

# Create models for both feature sets
num_classes = len(np.unique(y))
hidden_channels = 64

model_20 = GNNModel(X_train_20_scaled.shape[1], hidden_channels, num_classes).to(device)
model_full = GNNModel(X_train_full_scaled.shape[1], hidden_channels, num_classes).to(device)

print(f"Model for 20 features:")
print(model_20)
print(f"\nModel for all features:")
print(model_full)

# 6. Training Functions

Implement reusable training and evaluation functions with early stopping and model checkpointing.

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=100, patience=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    best_model = None
    patience_counter = 0
    
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        total_loss = 0
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.batch)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_train_loss = total_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                out = model(data.x, data.edge_index, data.batch)
                loss = criterion(out, data.y)
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model = copy.deepcopy(model)
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break
            
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")
    
    return best_model, train_losses, val_losses

def evaluate_model(model, loader):
    model.eval()
    predictions = []
    actual = []
    
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            pred = out.argmax(dim=1)
            predictions.extend(pred.cpu().numpy())
            actual.extend(data.y.cpu().numpy())
    
    predictions = np.array(predictions)
    actual = np.array(actual)
    
    results = {
        'accuracy': accuracy_score(actual, predictions),
        'precision': precision_score(actual, predictions, average='weighted'),
        'recall': recall_score(actual, predictions, average='weighted'),
        'f1': f1_score(actual, predictions, average='weighted'),
        'confusion_matrix': confusion_matrix(actual, predictions)
    }
    
    return results

# 7. Model Training and Evaluation (20 Features)

Train and evaluate the GNN model using only the top 20 features.

In [ ]:
print("Training model with top 20 features...")
model_20_best, train_losses_20, val_losses_20 = train_model(model_20, train_loader_20, val_loader_20)

# Evaluate on test set
results_20 = evaluate_model(model_20_best, test_loader_20)

print("\nResults for model with 20 features:")
print(f"Accuracy: {results_20['accuracy']:.4f}")
print(f"Precision: {results_20['precision']:.4f}")
print(f"Recall: {results_20['recall']:.4f}")
print(f"F1 Score: {results_20['f1']:.4f}")

# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses_20, label='Train Loss')
plt.plot(val_losses_20, label='Validation Loss')
plt.title('Training and Validation Loss (20 Features)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Save the model
torch.save({
    'model_state_dict': model_20_best.state_dict(),
    'results': results_20,
    'features': selected_features
}, '../playground/models/gnn_model_20.pt')

# 8. Model Training and Evaluation (All Features)

Train and evaluate the GNN model using all available features.

In [ ]:
print("Training model with all features...")
model_full_best, train_losses_full, val_losses_full = train_model(model_full, train_loader_full, val_loader_full)

# Evaluate on test set
results_full = evaluate_model(model_full_best, test_loader_full)

print("\nResults for model with all features:")
print(f"Accuracy: {results_full['accuracy']:.4f}")
print(f"Precision: {results_full['precision']:.4f}")
print(f"Recall: {results_full['recall']:.4f}")
print(f"F1 Score: {results_full['f1']:.4f}")

# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses_full, label='Train Loss')
plt.plot(val_losses_full, label='Validation Loss')
plt.title('Training and Validation Loss (All Features)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Save the model
torch.save({
    'model_state_dict': model_full_best.state_dict(),
    'results': results_full,
    'features': X_train.columns.tolist()
}, '../playground/models/gnn_model_full.pt')

# 9. Performance Comparison

Compare and visualize performance metrics between models trained on different feature sets.

In [ ]:
# Compare metrics
metrics = ['accuracy', 'precision', 'recall', 'f1']
model_names = ['20 Features', 'All Features']
values = [[results_20[m] for m in metrics],
          [results_full[m] for m in metrics]]

# Create comparison plot
plt.figure(figsize=(12, 6))
x = np.arange(len(metrics))
width = 0.35

plt.bar(x - width/2, values[0], width, label='20 Features')
plt.bar(x + width/2, values[1], width, label='All Features')

plt.xlabel('Metrics')
plt.ylabel('Score')
plt.title('Model Performance Comparison')
plt.xticks(x, metrics)
plt.legend()
plt.show()

# Print detailed comparison
print("\nDetailed Performance Comparison:")
print(f"{'Metric':<15} {'20 Features':<15} {'All Features':<15} {'Difference':<15}")
print("-" * 60)
for metric in metrics:
    diff = results_full[metric] - results_20[metric]
    print(f"{metric:<15} {results_20[metric]:<15.4f} {results_full[metric]:<15.4f} {diff:>+15.4f}")

# Save comparison results
comparison_results = {
    'top_20_features': {
        'metrics': results_20,
        'features': selected_features
    },
    'all_features': {
        'metrics': results_full,
        'features': X_train.columns.tolist()
    }
}

with open('../playground/models/gnn_comparison_results.json', 'w') as f:
    json.dump(comparison_results, f, indent=4)

# 10. Model Inference and Retraining Functions

Helper functions for model inference and retraining on new data.

In [ ]:
def load_model(model_path, num_features, device=device):
    """Load a saved GNN model"""
    checkpoint = torch.load(model_path)
    model = GNNModel(num_features, hidden_channels, num_classes).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    return model, checkpoint['features']

def prepare_data_for_inference(data, features, scaler):
    """Prepare data for model inference"""
    # Select features
    data = data[features]
    
    # Scale the data
    data_scaled = scaler.transform(data)
    
    # Convert to tensor
    x = torch.FloatTensor(data_scaled).to(device)
    
    # Create fully connected edges
    edge_index = []
    for i in range(len(data)):
        for j in range(len(data)):
            if i != j:
                edge_index.append([i, j])
    edge_index = torch.LongTensor(edge_index).t().to(device)
    
    return Data(x=x, edge_index=edge_index)

def predict(model, data):
    """Make predictions using the model"""
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index, None)
        pred = out.argmax(dim=1)
    return pred.cpu().numpy()

def retrain_model(model, new_data, features, scaler, old_data_ratio=0.3):
    """Retrain the model with new data while keeping some old data"""
    # Prepare new data
    X_new = new_data[features]
    y_new = new_data['Label']
    
    # Scale new data
    X_new_scaled = scaler.transform(X_new)
    
    # Create dataset and loader
    new_dataset = NetworkFlowDataset(X_new_scaled, y_new.values)
    new_loader = DataLoader(new_dataset, batch_size=1, shuffle=True)
    
    # Split new data for validation
    train_size = int(0.8 * len(new_loader))
    val_size = len(new_loader) - train_size
    
    train_loader, val_loader = torch.utils.data.random_split(
        new_loader, [train_size, val_size]
    )
    
    # Retrain model
    model_retrained, train_losses, val_losses = train_model(
        model, train_loader, val_loader, num_epochs=50, patience=5
    )
    
    return model_retrained

print("Helper functions for inference and retraining have been defined.")

# Example Usage

Here's how to use the trained models for inference and retraining:

In [ ]:
# Example: Load and use the 20-feature model
model_path = '../playground/models/gnn_model_20.pt'

if os.path.exists(model_path):
    # Load the model
    loaded_model, features = load_model(model_path, len(selected_features))
    print(f"Model loaded successfully with {len(features)} features")
    
    # Example: Make predictions on test data
    test_data = prepare_data_for_inference(X_test.iloc[:10], features, scaler_20)
    predictions = predict(loaded_model, test_data)
    print("\nExample predictions:", predictions)
    
    # Example: Retrain model with new data
    print("\nExample of model retraining:")
    retrained_model = retrain_model(loaded_model, df_processed.sample(1000), 
                                  features, scaler_20)
    print("Model retrained successfully")
else:
    print("Please run the training cells first to create the model file")